# Implementing the SVD from $A^T A$

I build the SVD in four steps:

1. form $A^T A$ and diagonalize it with `numpy.linalg.eigh`;
2. sort the eigenvalues in descending order to get $V$ and the singular values $\sigma_i = \sqrt{\lambda_i}$;
3. form $U$ from $A V / \sigma$ on the range of $A$, then complete it to an orthonormal basis with a QR factorization;
4. assemble $\Sigma$ as an $m \times n$ diagonal matrix.

The signs of singular vectors are not unique, so reconstruction and orthogonality are the right checks, not entrywise agreement with NumPy.


In [1]:
import numpy as np


def mySVD(A):
    """Return U, Sigma, V such that A = U @ Sigma @ V.T."""
    A = np.asarray(A, dtype=float)
    m, n = A.shape

    eigenvalues, eigenvectors = np.linalg.eigh(A.T @ A)
    order = np.argsort(eigenvalues)[::-1]
    V = eigenvectors[:, order]
    singular_values = np.sqrt(np.clip(eigenvalues[order], 0, None))

    # Eigenvalues that are numerically zero produce spurious singular values
    # of size sqrt(eps); use a relative threshold to determine the rank.
    tol = np.sqrt(np.finfo(float).eps) * singular_values[0]
    rank = int(np.sum(singular_values > tol))

    if rank:
        U, _ = np.linalg.qr(A @ V[:, :rank], mode="complete")
        # Make the sign of each column consistent with A V.
        for i in range(rank):
            if U[:, i] @ (A @ V[:, i]) < 0:
                U[:, i] *= -1
    else:
        U = np.eye(m)

    Sigma = np.zeros((m, n))
    Sigma[:rank, :rank] = np.diag(singular_values[:rank])
    return U, Sigma, V


for shape in [(3, 2), (2, 3), (4, 4), (5, 3), (3, 5)]:
    A = np.random.default_rng(2024).standard_normal(shape)
    U, Sigma, V = mySVD(A)
    reconstruction = np.linalg.norm(A - U @ Sigma @ V.T)
    orthogonality_u = np.linalg.norm(U.T @ U - np.eye(shape[0]))
    orthogonality_v = np.linalg.norm(V.T @ V - np.eye(shape[1]))
    print(f"{shape}: rank={np.linalg.matrix_rank(A)}, "
          f"reconstruction={reconstruction:.2e}, "
          f"||U^T U - I||={orthogonality_u:.2e}, "
          f"||V^T V - I||={orthogonality_v:.2e}")


(3, 2): rank=2, reconstruction=5.87e-16, ||U^T U - I||=3.55e-16, ||V^T V - I||=1.57e-16
(2, 3): rank=2, reconstruction=1.56e-15, ||U^T U - I||=3.60e-16, ||V^T V - I||=4.03e-16
(4, 4): rank=4, reconstruction=3.14e-15, ||U^T U - I||=7.02e-16, ||V^T V - I||=1.13e-15
(5, 3): rank=3, reconstruction=1.67e-15, ||U^T U - I||=7.13e-16, ||V^T V - I||=9.22e-16
(3, 5): rank=3, reconstruction=1.50e-15, ||U^T U - I||=5.93e-16, ||V^T V - I||=8.91e-16


In [2]:
A = np.array([[3.0, -2.0],
              [1.0, -4.0]])
U, Sigma, V = mySVD(A)
U_np, s_np, Vt_np = np.linalg.svd(A)

print("my singular values: ", np.diag(Sigma))
print("numpy singular values:", s_np)
print("reconstruction error:", np.linalg.norm(A - U @ Sigma @ V.T))
print("U =\n", U)
print("Sigma =\n", Sigma)
print("V =\n", V)


my singular values:  [5.11667274 1.95439508]
numpy singular values: [5.11667274 1.95439508]
reconstruction error: 8.881784197001252e-16
U =
 [[-0.64074744 -0.76775173]
 [-0.76775173  0.64074744]]
Sigma =
 [[5.11667274 0.        ]
 [0.         1.95439508]]
V =
 [[-0.52573111 -0.85065081]
 [ 0.85065081 -0.52573111]]
